## Берем датасет Reuters-21578 и достаем из него заголовки                                   

In [27]:
import numpy as np
import pandas as pd

In [ ]:
import html
import nltk
from nltk.corpus import reuters

nltk.download('reuters')

def load_reuters(split):
    ids = [f for f in reuters.fileids() if f.startswith(split + '/')]
    # ids = ['earn', 'acq', 'crude', 'trade', 'money-fx', 'interest', 'ship']
    texts = [html.unescape(reuters.raw(f)) for f in ids]
    labels = [reuters.categories(f) for f in ids]
    return texts, labels

def clean_text(text):
    text = html.unescape(text)
    title, _, body = text.partition('\n')
    text = f'{title.strip()}. {body}'
    text = TICKER.sub(' ', text)
    text = re.sub(r'[<>]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()

[nltk_data] Downloading package reuters to /Users/a11111/nltk_data...
[nltk_data]   Package reuters is already up-to-date!


In [ ]:
import re
TICKER = re.compile(r'<[A-Z]{1,5}(?:\.[A-Z]{1,3})*>')

text_train = [clean_text(t) for t in text_train]
text_test  = [clean_text(t) for t in text_test]

def single_label(texts, labels):
    pairs = [(t, l[0]) for t, l in zip(texts, labels) if len(l) == 1]
    return [t for t, _ in pairs], [l for _, l in pairs]

text_train1, tags_train1 = single_label(X_train, y_train)
text_test1,  tags_test1  = single_label(X_test, y_test)

In [35]:
print(len(text_train), len(text_test))

7769 3019


## Получение эмбеддингов (Bag Of Words)

In [38]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# матрица BoW
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(text_train1)

bow_df = pd.DataFrame(text_train1,
                     columns=vectorizer.get_feature_names_out(),
                     index=[f"{i+1}" for i in range(len(text_train1))])

print("BOW матрица:")
print(bow_df.round(3))

ValueError: Shape of passed values is (6577, 1), indices imply (6577, 23815)

In [41]:
vectorizer.get_feature_names_out(text_train1)

array(['00', '000', '0000', ..., 'zuyuan', 'zy', 'zzzz'],
      shape=(23815,), dtype=object)

In [42]:
from sklearn.feature_extraction.text import CountVectorizer

# Полный BoW без фильтрации словаря. Матрица хранится в разреженном формате CSR:
# в памяти лежат только ненулевые счётчики (~0.3% ячеек), поэтому весь корпус занимает единицы МБ.
# Все модели sklearn (TruncatedSVD, LogisticRegression, KNN, KMeans...) принимают такую матрицу напрямую.
vectorizer = CountVectorizer()
X_train_bow = vectorizer.fit_transform(text_train1)   # fit только на train
X_test_bow  = vectorizer.transform(text_test1)        # слова, которых нет в словаре train, отбрасываются
vocab = vectorizer.get_feature_names_out()

n_cells = X_train_bow.shape[0] * X_train_bow.shape[1]
sparse_mb = (X_train_bow.data.nbytes + X_train_bow.indices.nbytes + X_train_bow.indptr.nbytes) / 2**20
print('train:', X_train_bow.shape, ' test:', X_test_bow.shape)
print(f'ненулевых: {X_train_bow.nnz} из {n_cells} ({X_train_bow.nnz / n_cells:.2%})')
print(f'разреженная матрица: {sparse_mb:.1f} МБ, плотная int64: {n_cells * 8 / 2**30:.2f} ГБ')

train: (6577, 23815)  test: (2583, 23815)
ненулевых: 444784 из 156631255 (0.28%)
разреженная матрица: 5.1 МБ, плотная int64: 1.17 ГБ


In [43]:
# Табличный вид BoW: разреженный DataFrame, колонки — слова, строки — документы.
# Обычный pd.DataFrame(X.toarray()) развернул бы все нули и занял ~1.2 ГБ.
bow_df = pd.DataFrame.sparse.from_spmatrix(
    X_train_bow,
    columns=vocab,
    index=[f'{i+1}' for i in range(X_train_bow.shape[0])],
)
print(bow_df.shape, f'{bow_df.memory_usage().sum() / 2**20:.1f} МБ')

# слова первого документа по убыванию частоты
doc = bow_df.iloc[0]
doc[doc > 0].sort_values(ascending=False).head(15)

(6577, 23815) 5.1 МБ


and      23
at       20
the      18
dlrs     14
to       14
for      10
new       9
were      9
york      8
of        8
sales     7
times     7
cocoa     7
in        6
dec       6
Name: 1, dtype: Sparse[int64, 0]

In [ ]:
# Если всё же нужна плотная матрица (numpy-операции, методы без поддержки sparse):
# счётчики маленькие, поэтому хватает uint8 — 1 байт на ячейку вместо 8 (~150 МБ вместо ~1.2 ГБ).
assert X_train_bow.max() <= np.iinfo(np.uint8).max
X_train_dense = X_train_bow.astype(np.uint8).toarray()
print(X_train_dense.shape, f'{X_train_dense.nbytes / 2**20:.0f} МБ')